In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from accelerate import Accelerator
from torch_ema import ExponentialMovingAverage

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_name = "stabilityai/stable-diffusion-2-1-base"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    subfolder="tokenizer",
)

In [ ]:
#To run the pipeline, Specify the path to the training images and update the dictionary with id of classes to train on and their corresponding stable diffusion prompt

dataset_root = ''#path to the cropped image files(produced by the image cropper notebook)

#Dictonary of classes and corresponding stable diffusion prompt
classes = {
    "55": "a used teabag",
    "6": "used latex gloves",
    "77": "shredded paper",
    #Add more classes if desired
}

In [ ]:
#Data structure for loading cropped images
class WasteDataset(Dataset):
    def __init__(self, root_dir, classes, tokenizer, image_size=512):
        self.samples = []
        self.tokenizer = tokenizer
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

        for class_name, prompt in classes.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                self.samples.append((img_path, prompt))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, prompt = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        inputs = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "pixel_values": image,
            "input_ids": inputs.input_ids.squeeze(),
            "attention_mask": inputs.attention_mask.squeeze(),
        }

In [ ]:
dataset = WasteDataset(
    root_dir=dataset_root,
    classes=classes,
    tokenizer=tokenizer,
    image_size=512,
)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=4)

# Load the Stable Diffusion pipeline
pipeline = StableDiffusionPipeline.from_pretrained(model_name)
pipeline = pipeline.to(device)

# Extract model components
unet = pipeline.unet
vae = pipeline.vae
text_encoder = pipeline.text_encoder

# Freeze the VAE
vae.requires_grad_(False)

# Prepare for training
accelerator = Accelerator(
    gradient_accumulation_steps=1,
)

optimizer = torch.optim.AdamW(
    list(unet.parameters()) + list(text_encoder.parameters()),
    lr=1e-5,
    weight_decay=0.01,
)



In [ ]:
from tqdm.auto import tqdm
num_epochs = 15

# Prepare the learning rate scheduler
num_training_steps = num_epochs * len(dataloader)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0.1 * num_training_steps,
    num_training_steps=num_training_steps,
)

# Prepare everything with Accelerator
unet, text_encoder, optimizer, dataloader, lr_scheduler = accelerator.prepare(
    unet, text_encoder, optimizer, dataloader, lr_scheduler
)

# Initialize EMA
ema_unet = ExponentialMovingAverage(unet.parameters(), decay=0.999)


# Training loop
for epoch in range(num_epochs):
    unet.train()
    text_encoder.train()
    for step, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        with accelerator.accumulate(unet):
            # Move inputs to device
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # Check for NaNs or Infs in inputs
            if torch.isnan(pixel_values).any() or torch.isinf(pixel_values).any():
                print("NaN or Inf detected in pixel_values")
                continue

            with accelerator.autocast():
                # Convert images to latent space
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215  # Scaling factor

                # Sample noise
                noise = torch.randn_like(latents)
                timesteps = torch.randint(0, pipeline.scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()

                # Add noise to the latents
                noisy_latents = pipeline.scheduler.add_noise(latents, noise, timesteps)

                # Get text embeddings
                encoder_hidden_states = text_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )[0]

                # Predict the noise residual
                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

                # Compute loss
                if pipeline.scheduler.config.prediction_type == 'epsilon':
                    target = noise
                elif pipeline.scheduler.config.prediction_type == 'v_prediction':
                    target = pipeline.scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"Unknown prediction type {pipeline.scheduler.config.prediction_type}")

                loss = torch.nn.functional.mse_loss(model_pred, target, reduction="mean")

            # Check for NaNs or Infs in loss
            if torch.isnan(loss) or torch.isinf(loss):
                print("Loss is NaN or Inf!")
                continue

            accelerator.backward(loss)

            # Gradient clipping
            accelerator.clip_grad_norm_(unet.parameters(), max_norm=1.0)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            ema_unet.update()

            # Log loss
            if step % 10 == 0:
                print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.6f}")

    # Generate a sample image at the end of each epoch
    ema_unet.copy_to(unet.parameters())
    pipeline.unet = accelerator.unwrap_model(unet)
    pipeline.text_encoder = accelerator.unwrap_model(text_encoder)

    with torch.no_grad():
        with torch.autocast(device.type):
            sample_image = pipeline("a used teabag").images[0]
    sample_image.save(f"sample_epoch_{epoch+1}.png")

    # Save the model every epoch
    #accelerator.wait_for_everyone()
    #pipeline.unet.save_pretrained(f"fine_tuned_unet_epoch_{epoch+1}")

# Save the final model
accelerator.wait_for_everyone()
pipeline.unet.save_pretrained("fine_tuned_unet")


Epoch 1/15:   0%|          | 0/489 [00:00<?, ?it/s]

Epoch 1, Step 1, Loss: 0.024546
Epoch 1, Step 11, Loss: 0.033884
Epoch 1, Step 21, Loss: 0.122360
Epoch 1, Step 31, Loss: 0.022122
Epoch 1, Step 41, Loss: 0.307302
Epoch 1, Step 51, Loss: 0.124006
Epoch 1, Step 61, Loss: 0.024629
Epoch 1, Step 71, Loss: 0.019091
Epoch 1, Step 81, Loss: 0.132952
Epoch 1, Step 91, Loss: 0.111743
Epoch 1, Step 101, Loss: 0.084446
Epoch 1, Step 111, Loss: 0.131197
Epoch 1, Step 121, Loss: 0.008785
Epoch 1, Step 131, Loss: 0.059709
Epoch 1, Step 141, Loss: 0.055048
Epoch 1, Step 151, Loss: 0.096134
Epoch 1, Step 161, Loss: 0.078758
Epoch 1, Step 171, Loss: 0.145378
Epoch 1, Step 181, Loss: 0.143882
Epoch 1, Step 191, Loss: 0.053990
Epoch 1, Step 201, Loss: 0.005526
Epoch 1, Step 211, Loss: 0.024259
Epoch 1, Step 221, Loss: 0.002678
Epoch 1, Step 231, Loss: 0.065174
Epoch 1, Step 241, Loss: 0.036452
Epoch 1, Step 251, Loss: 0.144029
Epoch 1, Step 261, Loss: 0.011107
Epoch 1, Step 271, Loss: 0.129289
Epoch 1, Step 281, Loss: 0.047019
Epoch 1, Step 291, Loss: 

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2/15:   0%|          | 0/489 [00:00<?, ?it/s]

Epoch 2, Step 1, Loss: 0.047367
Epoch 2, Step 11, Loss: 0.066325
Epoch 2, Step 21, Loss: 0.005915
Epoch 2, Step 31, Loss: 0.033505
Epoch 2, Step 41, Loss: 0.179499
Epoch 2, Step 51, Loss: 0.144564
Epoch 2, Step 61, Loss: 0.049597
Epoch 2, Step 71, Loss: 0.036435
Epoch 2, Step 81, Loss: 0.116674
Epoch 2, Step 91, Loss: 0.034906
Epoch 2, Step 101, Loss: 0.003427
Epoch 2, Step 111, Loss: 0.018675
Epoch 2, Step 121, Loss: 0.203922
Epoch 2, Step 131, Loss: 0.040956
Epoch 2, Step 141, Loss: 0.135039
Epoch 2, Step 151, Loss: 0.191084
Epoch 2, Step 161, Loss: 0.067166
Epoch 2, Step 171, Loss: 0.203557
Epoch 2, Step 181, Loss: 0.144194
Epoch 2, Step 191, Loss: 0.007568
Epoch 2, Step 201, Loss: 0.090957
Epoch 2, Step 211, Loss: 0.010613
Epoch 2, Step 221, Loss: 0.033238
Epoch 2, Step 231, Loss: 0.218530
Epoch 2, Step 241, Loss: 0.005398
Epoch 2, Step 251, Loss: 0.092282
Epoch 2, Step 261, Loss: 0.515317
Epoch 2, Step 271, Loss: 0.111266
Epoch 2, Step 281, Loss: 0.095386
Epoch 2, Step 291, Loss: 

...

Epoch 14, Step 1, Loss: 0.226244
Epoch 14, Step 11, Loss: 0.036524
Epoch 14, Step 21, Loss: 0.194756
Epoch 14, Step 31, Loss: 0.098970
Epoch 14, Step 41, Loss: 0.101395
Epoch 14, Step 51, Loss: 0.144498
Epoch 14, Step 61, Loss: 0.038387
Epoch 14, Step 71, Loss: 0.155706
Epoch 14, Step 81, Loss: 0.159579
Epoch 14, Step 91, Loss: 0.076956
Epoch 14, Step 101, Loss: 0.002508
Epoch 14, Step 111, Loss: 0.084211
Epoch 14, Step 121, Loss: 0.003039
Epoch 14, Step 131, Loss: 0.120319
Epoch 14, Step 141, Loss: 0.098169
Epoch 14, Step 151, Loss: 0.027495
Epoch 14, Step 161, Loss: 0.099504
Epoch 14, Step 171, Loss: 0.065963
Epoch 14, Step 181, Loss: 0.038280
Epoch 14, Step 191, Loss: 0.255108
Epoch 14, Step 201, Loss: 0.148799
Epoch 14, Step 211, Loss: 0.002688
Epoch 14, Step 221, Loss: 0.053347
Epoch 14, Step 231, Loss: 0.011929
Epoch 14, Step 241, Loss: 0.054180
Epoch 14, Step 251, Loss: 0.042453
Epoch 14, Step 261, Loss: 0.367308
Epoch 14, Step 271, Loss: 0.358843
Epoch 14, Step 281, Loss: 0.011

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 15/15:   0%|          | 0/489 [00:00<?, ?it/s]

Epoch 15, Step 1, Loss: 0.219089
Epoch 15, Step 11, Loss: 0.011531
Epoch 15, Step 21, Loss: 0.208367
Epoch 15, Step 31, Loss: 0.008949
Epoch 15, Step 41, Loss: 0.226353
Epoch 15, Step 51, Loss: 0.004384
Epoch 15, Step 61, Loss: 0.005428
Epoch 15, Step 71, Loss: 0.023864
Epoch 15, Step 81, Loss: 0.320515
Epoch 15, Step 91, Loss: 0.013249
Epoch 15, Step 101, Loss: 0.003921
Epoch 15, Step 111, Loss: 0.089240
Epoch 15, Step 121, Loss: 0.015039
Epoch 15, Step 131, Loss: 0.192146
Epoch 15, Step 141, Loss: 0.143765
Epoch 15, Step 151, Loss: 0.092917
Epoch 15, Step 161, Loss: 0.206431
Epoch 15, Step 171, Loss: 0.202686
Epoch 15, Step 181, Loss: 0.129259
Epoch 15, Step 191, Loss: 0.061379
Epoch 15, Step 201, Loss: 0.217908
Epoch 15, Step 211, Loss: 0.022768
Epoch 15, Step 221, Loss: 0.004182
Epoch 15, Step 231, Loss: 0.023115
Epoch 15, Step 241, Loss: 0.123445
Epoch 15, Step 251, Loss: 0.028941
Epoch 15, Step 261, Loss: 0.072530
Epoch 15, Step 271, Loss: 0.298046
Epoch 15, Step 281, Loss: 0.280

  0%|          | 0/50 [00:00<?, ?it/s]

# Using the tuned model to generate image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pretrained pipeline
model_name = "stabilityai/stable-diffusion-2-1-base"
pipeline = StableDiffusionPipeline.from_pretrained(model_name).to(device)

In [ ]:
# Load the fine-tuned U-Net
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
fine_tuned_unet_path = "/content/fine_tuned_unet"
pipeline.unet = UNet2DConditionModel.from_pretrained(
    fine_tuned_unet_path,
    subfolder=None,
).to(device)


In [ ]:
def generate_image(class_name, negative_prompt=None, num_inference_steps=50, guidance_scale=7.5):
    prompt = classes.get(class_name)
    if prompt is None:
        raise ValueError(f"Class '{class_name}' not found in classes dictionary.")

    with torch.no_grad():
        with torch.autocast(device.type):
            image = pipeline(
                prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
            ).images[0]
    return image


In [ ]:
# Example usage

class_name = "55"  # Replace with the desired class
image = generate_image(class_name)
image.save(f"generated_{class_name}.png")

  0%|          | 0/50 [00:00<?, ?it/s]